# Cleaning the AML dataset

In [3]:
# in terminal: uv sync
import pandas as pd
import numpy as np
import re

In [4]:
# Loading the datasets

accounts = pd.read_csv('../../data/raw/HI-Small_accounts.csv')
trans = pd.read_csv('../../data/raw/HI-Small_Trans.csv')

In [ ]:
accounts.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [ ]:
accounts.describe()

,Bank ID
count,518581.000000
mean,103425.510831
std,121479.549761
min,1.000000
25%,11107.000000
50%,32014.000000
75%,217096.000000
max,356303.000000


In [5]:
trans.head()


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


## Atul's part

Cleaning tasks:
* separating the date-time column adn formatting it correctlu
* inserting two columns (`amount_received_euro` and `amount_paid_euro`) with the change in currency
* other--- 

In [43]:
#Separating Date and time
trans['Timestamp'] = pd.to_datetime(trans['Timestamp'], format='%Y/%m/%d %H:%M')
trans['Date'] = trans['Timestamp'].dt.date
trans['Time'] = trans['Timestamp'].dt.time

#check time format
trans["Time"].unique()[:10]

#Creating separate hour and minute columns for confirmation and cleaning check
trans["hour"] = trans["Time"].apply(lambda x: x.hour)
trans["minute"] = trans["Time"].apply(lambda x: x.minute)

trans.head(20)



,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,pay_curr_Mexican Peso,pay_curr_Ruble,pay_curr_Rupee,pay_curr_Saudi Riyal,pay_curr_Shekel,pay_curr_Swiss Franc,pay_curr_UK Pound,pay_curr_US Dollar,pay_curr_Yen,pay_curr_Yuan
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,...,0,0,0,0,0,0,0,1,0,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,...,0,0,0,0,0,0,0,1,0,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,...,0,0,0,0,0,0,0,1,0,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,...,0,0,0,0,0,0,0,1,0,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,...,0,0,0,0,0,0,0,1,0,0
5,2022-09-01 00:03:00,1,8000F5AD0,1,8000F5AD0,6162.44,US Dollar,6162.44,US Dollar,Reinvestment,...,0,0,0,0,0,0,0,1,0,0
6,2022-09-01 00:08:00,1,8000EBAC0,1,8000EBAC0,14.26,US Dollar,14.26,US Dollar,Reinvestment,...,0,0,0,0,0,0,0,1,0,0
7,2022-09-01 00:16:00,1,8000EC1E0,1,8000EC1E0,11.86,US Dollar,11.86,US Dollar,Reinvestment,...,0,0,0,0,0,0,0,1,0,0
8,2022-09-01 00:26:00,12,8000EC280,2439,8017BF800,7.66,US Dollar,7.66,US Dollar,Credit Card,...,0,0,0,0,0,0,0,1,0,0
9,2022-09-01 00:21:00,1,8000EDEC0,211050,80AEF5310,383.71,US Dollar,383.71,US Dollar,Credit Card,...,0,0,0,0,0,0,0,1,0,0


In [46]:
#Creating hot-coding for each currency (Long to wide) for Receiving currency and Payment Currency
receiving_dummies = pd.get_dummies(
    trans["Receiving Currency"],
    prefix="rec_curr"
).astype(int)

payment_dummies = pd.get_dummies(
    trans["Payment Currency"],
    prefix="pay_curr"
).astype(int)

#adding new numeric columns to our trans DF
trans = pd.concat(
    [trans, receiving_dummies, payment_dummies],
    axis=1
)

In [ ]:
#Check the conversion specifically for for receiving and payment made not in US (Not important)
#not_us_dollar = trans[trans['rec_curr_US Dollar'] == 0]
#pd.set_option('display.max_columns', None)
#not_us_dollar.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,Date,Time,hour,minute,rec_curr_Australian Dollar,rec_curr_Bitcoin,rec_curr_Brazil Real,rec_curr_Canadian Dollar,rec_curr_Euro,rec_curr_Mexican Peso,rec_curr_Ruble,rec_curr_Rupee,rec_curr_Saudi Riyal,rec_curr_Shekel,rec_curr_Swiss Franc,rec_curr_UK Pound,rec_curr_US Dollar,rec_curr_Yen,rec_curr_Yuan,pay_curr_Australian Dollar,pay_curr_Bitcoin,pay_curr_Brazil Real,pay_curr_Canadian Dollar,pay_curr_Euro,pay_curr_Mexican Peso,pay_curr_Ruble,pay_curr_Rupee,pay_curr_Saudi Riyal,pay_curr_Shekel,pay_curr_Swiss Franc,pay_curr_UK Pound,pay_curr_US Dollar,pay_curr_Yen,pay_curr_Yuan
1155,2022-09-01 00:16:00,220,8001C8C51,1420,8003093C1,0.025852,Bitcoin,0.025852,Bitcoin,Bitcoin,0,2022-09-01,00:16:00,0,16,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1173,2022-09-01 00:22:00,1362,80030A870,1362,80030A870,52.110000,Euro,61.060000,US Dollar,ACH,0,2022-09-01,00:22:00,0,22,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1174,2022-09-01 00:22:00,1362,80030A870,11,80064C9B0,52.110000,Euro,52.110000,Euro,Credit Card,0,2022-09-01,00:22:00,0,22,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1467,2022-09-01 00:15:00,1,80005C0A1,1588,8003AC471,0.016891,Bitcoin,0.016891,Bitcoin,Bitcoin,0,2022-09-01,00:15:00,0,15,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3940,2022-09-01 00:08:00,513,8006538E1,1688,8006DD361,1.621978,Bitcoin,1.621978,Bitcoin,Bitcoin,0,2022-09-01,00:08:00,0,8,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
#Column for currency mismatch between receiving and payment
trans["currency_mismatch"] = (
    trans["Payment Currency"] != trans["Receiving Currency"]
).astype(int)


In [ ]:
#checking if column is made correctly with 1 values (not that important)
#curr_mismatch = trans[trans["currency_mismatch"] == 1]
#curr_mismatch.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,Date,Time,hour,minute,rec_curr_Australian Dollar,rec_curr_Bitcoin,rec_curr_Brazil Real,rec_curr_Canadian Dollar,rec_curr_Euro,rec_curr_Mexican Peso,rec_curr_Ruble,rec_curr_Rupee,rec_curr_Saudi Riyal,rec_curr_Shekel,rec_curr_Swiss Franc,rec_curr_UK Pound,rec_curr_US Dollar,rec_curr_Yen,rec_curr_Yuan,pay_curr_Australian Dollar,pay_curr_Bitcoin,pay_curr_Brazil Real,pay_curr_Canadian Dollar,pay_curr_Euro,pay_curr_Mexican Peso,pay_curr_Ruble,pay_curr_Rupee,pay_curr_Saudi Riyal,pay_curr_Shekel,pay_curr_Swiss Franc,pay_curr_UK Pound,pay_curr_US Dollar,pay_curr_Yen,pay_curr_Yuan,currency_mismatch
1173,2022-09-01 00:22:00,1362,80030A870,1362,80030A870,52.11,Euro,61.06,US Dollar,ACH,0,2022-09-01,00:22:00,0,22,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
7156,2022-09-01 00:28:00,11318,800C51010,11318,800C51010,76.06,Euro,89.12,US Dollar,ACH,0,2022-09-01,00:28:00,0,28,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
7925,2022-09-01 00:12:00,795,800D98770,795,800D98770,17.69,Australian Dollar,12.52,US Dollar,ACH,0,2022-09-01,00:12:00,0,12,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
8467,2022-09-01 00:01:00,1047,800E92CF0,1047,800E92CF0,19.43,Euro,22.77,US Dollar,ACH,0,2022-09-01,00:01:00,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
11529,2022-09-01 00:22:00,11157,80135FFC0,11157,80135FFC0,98.34,Euro,115.24,US Dollar,ACH,0,2022-09-01,00:22:00,0,22,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1


In [54]:
#Now converting payment format (Hot-Coding; long to wide)
trans["Payment Format"].value_counts()

payment_format_dummies = pd.get_dummies(
    trans["Payment Format"],
    prefix="pay_for"
).astype(int)

#adding new numeric column to our DF
trans =pd.concat(
    [trans, payment_format_dummies],
    axis=1
)

trans.head(20)


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,Date,Time,hour,minute,rec_curr_Australian Dollar,rec_curr_Bitcoin,rec_curr_Brazil Real,rec_curr_Canadian Dollar,rec_curr_Euro,rec_curr_Mexican Peso,rec_curr_Ruble,rec_curr_Rupee,rec_curr_Saudi Riyal,rec_curr_Shekel,rec_curr_Swiss Franc,rec_curr_UK Pound,rec_curr_US Dollar,rec_curr_Yen,rec_curr_Yuan,pay_curr_Australian Dollar,pay_curr_Bitcoin,pay_curr_Brazil Real,pay_curr_Canadian Dollar,pay_curr_Euro,pay_curr_Mexican Peso,pay_curr_Ruble,pay_curr_Rupee,pay_curr_Saudi Riyal,pay_curr_Shekel,pay_curr_Swiss Franc,pay_curr_UK Pound,pay_curr_US Dollar,pay_curr_Yen,pay_curr_Yuan,currency_mismatch,pay_for_ACH,pay_for_Bitcoin,pay_for_Cash,pay_for_Cheque,pay_for_Credit Card,pay_for_Reinvestment,pay_for_Wire
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0,2022-09-01,00:20:00,0,20,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0,2022-09-01,00:20:00,0,20,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0,2022-09-01,00:00:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0,2022-09-01,00:02:00,0,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0,2022-09-01,00:06:00,0,6,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
5,2022-09-01 00:03:00,1,8000F5AD0,1,8000F5AD0,6162.44,US Dollar,6162.44,US Dollar,Reinvestment,0,2022-09-01,00:03:00,0,3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
6,2022-09-01 00:08:00,1,8000EBAC0,1,8000EBAC0,14.26,US Dollar,14.26,US Dollar,Reinvestment,0,2022-09-01,00:08:00,0,8,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
7,2022-09-01 00:16:00,1,8000EC1E0,1,8000EC1E0,11.86,US Dollar,11.86,US Dollar,Reinvestment,0,2022-09-01,00:16:00,0,16,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0
8,2022-09-01 00:26:00,12,8000EC280,2439,8017BF800,7.66,US Dollar,7.66,US Dollar,Credit Card,0,2022-09-01,00:26:00,0,26,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
9,2022-09-01 00:21:00,1,8000EDEC0,211050,80AEF5310,383.71,US Dollar,383.71,US Dollar,Credit Card,0,2022-09-01,00:21:00,0,21,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0


#Check the conversion specifically for for receiving and payment made not in US (Not important)
#not_us_dollar = trans[trans['rec_curr_US Dollar'] == 0]
#pd.set_option('display.max_columns', None)
#not_us_dollar.head()